In [19]:
import numpy as np
import pandas as pd
import time
import logging
import sys
import json
from datetime import datetime
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import xgboost as xgb

# --- 1. 配置与环境设置 ---
def setup_logging(log_file='training_xgboost_regression.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger()
    root_logger.handlers = []
    root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w')
    file_handler.setFormatter(log_formatter)
    root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(log_formatter)
    root_logger.addHandler(console_handler)

np.random.seed(42)

def get_regression_data(test_size=0.3):
    """准备Boston房价回归数据集"""
    logging.info("正在准备Boston房价回归数据集...")
    
    # 加载Boston房价数据集
    df = pd.read_csv('/kaggle/input/the-boston-houseprice-data/boston.csv')
    
    # 检查并处理数据
    if 'Unnamed: 0' in df.columns:
        df = df.drop('Unnamed: 0', axis=1)
    
    # 特征和目标变量
    X = df.drop(['MEDV'], axis=1)  # 特征矩阵
    y = df['MEDV']  # 目标变量（房价中位数）
    
    # 数据标准化
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # 对目标变量标准化
    y = (y - y.mean()) / y.std()
    
    # 划分训练测试集
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    logging.info("Boston房价回归数据集准备完毕。")
    logging.info(f"训练样本数: {len(X_train)}, 测试样本数: {len(X_test)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")
    
    return X_train, X_test, y_train, y_test

class XGBoostRegression:
    """XGBoost回归模型"""
    def __init__(self, params=None):
        if params is None:
            self.params = {
                'objective': 'reg:squarederror',
                'eval_metric': 'rmse',  # 这里保持rmse，但我们会记录mse
                'eta': 0.1,
                'max_depth': 6,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'seed': 42,
                'nthread': -1
            }
        else:
            self.params = params
        self.model = None
        self.evals_result = {}  # 新增属性来存储评估结果
    
    def fit(self, X_train, y_train, X_val, y_val, num_boost_round=100, early_stopping_rounds=10):
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_val, label=y_val)
        
        self.evals_result = {}  # 重置评估结果
        
        self.model = xgb.train(
            self.params,
            dtrain,
            num_boost_round=num_boost_round,
            evals=[(dtrain, 'train'), (dval, 'val')],
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False,
            evals_result=self.evals_result  # 直接收集评估结果
        )
        return self.model
    
    def predict(self, X):
        dtest = xgb.DMatrix(X)
        return self.model.predict(dtest)
    
    def evaluate(self, X, y):
        preds = self.predict(X)
        return mean_squared_error(y, preds)

def train_xgboost(X_train, X_test, y_train, y_test, hparams, log_filepath, model_save_path):
    # 进一步划分训练集和验证集
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=42
    )

    # 初始化模型
    xgb_model = XGBoostRegression(hparams['model_params'])

    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "hyperparameters": hparams,
        "training_mse_per_round": [],
        "val_mse_per_round": []
    }

    logging.info("="*30)
    logging.info("  启动 XGBoost 训练流程")
    logging.info("="*30)

    start_time = time.time()

    # 训练模型
    model = xgb_model.fit(
        X_train, y_train,
        X_val, y_val,
        num_boost_round=hparams['num_boost_round'],
        early_stopping_rounds=hparams['early_stopping_rounds']
    )

    # 获取训练过程中的评估结果
    train_mse = [x**2 for x in xgb_model.evals_result['train']['rmse']]  # 将rmse转换为mse
    val_mse = [x**2 for x in xgb_model.evals_result['val']['rmse']]  # 将rmse转换为mse

    # 记录日志
    for i, (train_err, val_err) in enumerate(zip(train_mse, val_mse)):
        log_data["training_mse_per_round"].append(train_err)
        log_data["val_mse_per_round"].append(val_err)

        if (i + 1) % 10 == 0:  # 每10轮打印一次日志
            logging.info(f"Round {i+1} | 训练MSE: {train_err:.4f} | 验证MSE: {val_err:.4f}")

    # 测试集评估
    test_mse = xgb_model.evaluate(X_test, y_test)
    test_rmse = np.sqrt(test_mse)

    log_data["final_test_rmse"] = test_rmse
    log_data["final_test_mse"] = test_mse

    logging.info(f"训练完成。最佳迭代次数: {model.best_iteration}")
    logging.info(f"测试集RMSE: {test_rmse:.4f}")
    logging.info(f"测试集MSE: {test_mse:.4f}")

    # 保存模型
    model.save_model(model_save_path)
    logging.info(f"模型已保存到 {model_save_path}")

    # 保存日志
    with open(log_filepath, 'w') as f:
        json.dump(log_data, f, indent=4)

    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")

    return test_mse

if __name__ == '__main__':
    # --- 文件与超参数定义 ---
    LOG_FILE = 'training_xgboost_regression.log'
    JSON_LOG_FILE = 'training_xgboost_log.json'
    MODEL_SAVE_PATH = 'xgboost_regression.model'
    
    HYPERPARAMETERS = {
        "num_boost_round": 200,
        "early_stopping_rounds": 20,
        "model_params": {
            'objective': 'reg:squarederror',
            'eval_metric': 'rmse',  # 内部仍使用rmse，但我们会转换为mse记录
            'eta': 0.1,
            'max_depth': 6,
            'subsample': 0.8,
            'colsample_bytree': 0.8,
            'alpha': 0.1,  # L1正则化
            'lambda': 1.0,  # L2正则化
            'seed': 42
        },
        "loss_target": 0.1
    }

    setup_logging(LOG_FILE)
    logging.info(f"回归实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    X_train, X_test, y_train, y_test = get_regression_data()
    
    test_mse = train_xgboost(
        X_train, X_test, y_train, y_test,
        hparams=HYPERPARAMETERS,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    # 验证保存的模型
    if test_mse < float('inf'):
        logging.info("\n" + "="*30)
        logging.info("  开始验证已保存的模型")
        logging.info("="*30)
        
        # 加载模型
        loaded_model = xgb.Booster()
        loaded_model.load_model(MODEL_SAVE_PATH)
        
        # 创建DMatrix
        dtest = xgb.DMatrix(X_test)
        
        # 预测
        preds = loaded_model.predict(dtest)
        
        # 计算MSE
        mse = mean_squared_error(y_test, preds)
        rmse = np.sqrt(mse)
        
        logging.info(f"加载模型的测试集RMSE: {rmse:.4f}")
        logging.info(f"加载模型的测试集MSE: {mse:.4f}")
        
        assert abs(mse - test_mse) < 1e-3, "验证失败！"
        logging.info("验证成功！模型可以被正确保存和复用。")
    else:
        logging.info("未能成功训练和保存模型。")

2025-08-18 08:37:02,446 [INFO ]  回归实验超参数: {
    "num_boost_round": 200,
    "early_stopping_rounds": 20,
    "model_params": {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "eta": 0.1,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "alpha": 0.1,
        "lambda": 1.0,
        "seed": 42
    },
    "loss_target": 0.1
}
2025-08-18 08:37:02,447 [INFO ]  正在准备Boston房价回归数据集...
2025-08-18 08:37:02,462 [INFO ]  Boston房价回归数据集准备完毕。
2025-08-18 08:37:02,463 [INFO ]  训练样本数: 354, 测试样本数: 152
2025-08-18 08:37:02,464 [INFO ]  特征维度: 13, 目标维度: 1
2025-08-18 08:37:02,467 [INFO ]  ==============================
2025-08-18 08:37:02,467 [INFO ]    启动 XGBoost 训练流程
2025-08-18 08:37:02,468 [INFO ]  ==============================
2025-08-18 08:37:02,681 [INFO ]  Round 10 | 训练MSE: 0.2203 | 验证MSE: 0.4189
2025-08-18 08:37:02,682 [INFO ]  Round 20 | 训练MSE: 0.0644 | 验证MSE: 0.3207
2025-08-18 08:37:02,683 [INFO ]  Round 30 | 训练MSE: 0.0251 | 验证

/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [08:37:02] WARNING: /workspace/src/c_api/c_api.cc:1240: Saving into deprecated binary model format, please consider using `json` or `ubj`. Model format will default to JSON in XGBoost 2.2 if not specified.
  warnings.warn(smsg, UserWarning)
